# The normal distribution
The PDF by hand and with scipy, two sliders for the parameters, and the CDF.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

## 1. The PDF formula by hand
Heights of adult men: mean 68 inches, standard deviation 3 inches.

In [ ]:
def normal_pdf(x, mu, sigma):
    # 1 / (sigma * sqrt(2 pi)) * e^(-(x - mu)^2 / (2 sigma^2))
    return np.exp(-((x - mu) ** 2) / (2 * sigma ** 2)) / (sigma * np.sqrt(2 * np.pi))

print(normal_pdf(68, 68, 3), normal_pdf(72, 68, 3))
# scipy gives the same numbers: loc = mu, scale = sigma
heights = stats.norm(loc=68, scale=3)
print(heights.pdf(68), heights.pdf(72))

## 2. What the two parameters do
Drag the sliders: the mean moves the curve, the standard deviation widens it. The grey curve, N(0, 1), stays for comparison.

In [ ]:
x = np.linspace(-10, 10, 400)

def figure_with_slider(values, label, make_curve):
    fig = go.Figure()
    fig.add_scatter(x=x, y=stats.norm(0, 1).pdf(x), line=dict(color="grey"), name="N(0, 1)")
    for v in values:
        fig.add_scatter(x=x, y=make_curve(v), visible=False, name=f"{label} = {v}")
    fig.data[1].visible = True
    steps = []
    for i, v in enumerate(values):
        vis = [True] + [j == i for j in range(len(values))]
        steps.append(dict(method="update", args=[{"visible": vis}], label=str(v)))
    fig.update_layout(sliders=[dict(steps=steps, currentvalue=dict(prefix=label + " = "))],
                      template="simple_white", yaxis_range=[0, 0.85])
    return fig

figure_with_slider([-4, -2, 0, 2, 4], "mean", lambda m: stats.norm(m, 1).pdf(x)).show()
figure_with_slider([0.5, 1, 2, 3], "std", lambda s: stats.norm(0, s).pdf(x)).show()

## 3. Building the formula term by term

In [ ]:
x = np.linspace(-5, 5, 400)
sigma = 1.5
steps = {
    "e^(-x^2)": np.exp(-x ** 2),
    "e^(-(x - 2)^2)": np.exp(-(x - 2) ** 2),
    "e^(-x^2 / (2 sigma^2))": np.exp(-x ** 2 / (2 * sigma ** 2)),
    "normal PDF": np.exp(-x ** 2 / (2 * sigma ** 2)) / (sigma * np.sqrt(2 * np.pi)),
}
fig = go.Figure()
for name, y in steps.items():
    fig.add_scatter(x=x, y=y, name=name)
fig.update_layout(template="simple_white")
fig.show()
# area under the unscaled bell, and under the PDF (trapezoid rule; the curves are tiny beyond +-5)
print(np.trapezoid(steps["e^(-x^2 / (2 sigma^2))"], x), sigma * np.sqrt(2 * np.pi))
print(np.trapezoid(steps["normal PDF"], x))

In [ ]:
# the classic integral behind the 1 / (sigma sqrt(2 pi)) term, checked numerically
from scipy import integrate
print(integrate.quad(lambda t: np.exp(-t ** 2 / 2), -np.inf, np.inf)[0], np.sqrt(2 * np.pi))
print(integrate.quad(lambda t: np.exp(-t ** 2 / (2 * 1.5 ** 2)), -np.inf, np.inf)[0], 1.5 * np.sqrt(2 * np.pi))

## 4. The CDF

In [ ]:
print(heights.cdf(68), heights.cdf(72), heights.sf(72))
x = np.linspace(-5, 5, 400)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=["PDF", "CDF"])
for mu, s in [(0, 0.5), (0, 1), (0, 2), (-2, 0.7)]:
    d = stats.norm(mu, s)
    fig.add_scatter(x=x, y=d.pdf(x), name=f"mu={mu}, sigma={s}", row=1, col=1)
    fig.add_scatter(x=x, y=d.cdf(x), showlegend=False, row=2, col=1)
fig.update_layout(template="simple_white", height=600)
fig.show()

## A narrow curve is a tall curve (section 3.1)
Newborn boys' length (WHO Child Growth Standards, at birth: median 49.8842 cm, coefficient of variation 0.03795) against adult men's height N(68, 3^2), in inches.

In [ ]:
import numpy as np
from scipy import stats

baby = stats.norm(49.8842 / 2.54, 49.8842 * 0.03795 / 2.54)
adult = stats.norm(68, 3)
for name, d in [('newborn', baby), ('adult', adult)]:
    m, s = d.mean(), d.std()
    print(name, 'mean', round(m, 2), 'sd', round(s, 3), 'peak', round(d.pdf(m), 3), '95% range', round(m - 2 * s, 1), round(m + 2 * s, 1))
print('area under e^(-x^2):', __import__('scipy').integrate.quad(lambda x: np.exp(-x ** 2), -np.inf, np.inf)[0], 'sqrt(pi) =', np.sqrt(np.pi))